### Spin Test 
This script projects data to surface space and performs a spin test

#### Import Packages

In [85]:
import numpy as np
import pandas as pd
import netneurotools.stats as stats
from neuromaps.datasets import fetch_atlas
from nilearn.datasets import fetch_atlas_schaefer_2018
from netneurotools.datasets import fetch_schaefer2018
import neuromaps
import nibabel as nib
from neuromaps.parcellate import Parcellater
from neuromaps import transforms
from neuromaps import nulls
from scipy.stats import pearsonr, spearmanr
from nilearn.plotting import plot_surf
from nilearn.surface import load_surf_data
from nilearn import surface, plotting, datasets
from neuromaps import datasets, images, nulls, resampling, stats
import netneurotools.datasets
from netneurotools.datasets import fetch_cammoun2012

#### Set Paths and Variables

In [52]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])?  y


In [148]:
dataset = 'longglucest_outputmeasures2'
atlas = 'atl-Cammoun2012_res-500' # 'Schaefer2018_1000Parcels_17Networks'
nmaps = ["NMDA", "mGluR5", "GABA"]
maps = ["cest", "NMDA", "mGluR5", "GABA"]
von_economos = ["association","limbic"]

threshold = '60'
grouped = False
n_permute = 100

# Set paths
inpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/" + atlas
outpath = "/Users/pecsok/Desktop/ImageData/PMACS_remote/data/nmaps/analyses/" + atlas
root = "/Users/pecsok/Library/CloudStorage/Box-Box/GluCEST PhD/Manuscripts/Neuromaps/Data/Curated"

# Read in data
#receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_scale1000_17.csv", sep=',')
#means= pd.read_csv(inpath + '/means_std_' + dataset + '_' + atlas + '.csv', sep=',') #subjectnmaps
means= pd.read_csv(root + '/means_df_all_curated_' + atlas + '_threshold' + threshold + '.csv')

# Import new VE names
new_VEnames = pd.read_csv('/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/data/renamed_VEregions.csv')
new_VEnames['VEregion'] = new_VEnames['VE_Region'].replace('primary sensory', 'primary/secondary sensory')
# Add parcel labels


In [169]:
new_VEnames["VEregion"].unique().tolist()

['primary/secondary sensory',
 'association2',
 'association1',
 'primary motor',
 'limbic',
 'insular']

<class 'pandas.core.frame.DataFrame'>


In [152]:
# Reformat data 
means["Parcel"] = means["Parcel"].str.replace(' NZMean', '', regex=False)
if 'grouped' == True: # Filter by hstatus group of interest if desired
    means = means[means['hstatus']== group] 
means.set_index('Parcel', inplace = True)
#print(means)
"""
receptor_df = receptor_df.reset_index(drop=True)
receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)
receptor_df.index = pd.RangeIndex(start=1, stop=len(receptor_df) + 1, step=1)
print(means)
print(receptor_df)
"""
if atlas == 'Schaefer2018_1000Parcels_17Networks':
    receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_scale1000_17.csv", sep=',')
    receptor_df = receptor_df.reset_index(drop=True)
    receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)
    receptor_df.index = pd.RangeIndex(start=1, stop=len(receptor_df) + 1, step=1)
    # Read in surface atlases
    schaefer_cifti_path = '/Users/pecsok/neuromaps-data/atlases/fsLR/Schaefer2018_900Parcels_17Networks_order.dlabel.nii'
    annot_rh_schaefer = '/Users/pecsok/neuromaps-data/atlases/fsaverage/lh.Schaefer2018_1000Parcels_17Networks_order.annot'
    annot_lh_schaefer = '/Users/pecsok/neuromaps-data/atlases/fsaverage/rh.Schaefer2018_1000Parcels_17Networks_order.annot'
    # Fetch atlas labels
    schaefer = fetch_atlas_schaefer_2018(n_rois=1000, yeo_networks=17)
    labels = schaefer['labels']  # These are the MNI coordinates of the parcels
    labels = [label.decode('utf-8') for label in labels]

elif atlas == 'atl-Cammoun2012_res-500':
    scale = 'cammoun2012_scale500' #'scale1000_17' #'Harvard_Oxford_sub' 
    # Import surface data
    cammoun = netneurotools.datasets.fetch_cammoun2012(version='fsaverage')
    cammoun = cammoun['scale500']
    annot_lh = cammoun[0]
    annot_rh = cammoun[1]
    # Import receptor_df
    receptor_df = pd.read_csv("/Users/pecsok/projects/Neuromaps/pecsok_pfns/neuromaps/results/receptor_data_cammoun2012_scale500.csv", sep=',')
    receptor_df.rename(columns={'GABAa': 'GABA'}, inplace=True)   
    receptor_df.index = pd.RangeIndex(start=1, stop=len(receptor_df) + 1, step=1)
    # Import labels and remove subcortex
    cam = fetch_cammoun2012()
    info = pd.read_csv(cam['info'], sep=',')
    info = info[info['scale']== 'scale500']
    #labels = info['label'][info['structure']=='cortex'].values
    hemisphere = info['hemisphere'][info['structure']=='cortex'].values
    labels = info['label'][info['structure']=='cortex'].values + hemisphere
    print(labels.shape)
    #print(labels)
    # Remove subcortex from other dfs
    receptor_df = receptor_df[receptor_df['Parcel'].str.contains('|'.join(labels))]
    # Add new VE region names based on Golia's divided association cortex map
    receptor_df['VEregion'] = region_names['VE_Region'].values
    print(receptor_df.shape)
    means = means[means.index.str.contains('|'.join(labels))]
    # Add VEregion to means_df
    new_VEnames['Parcel'] = labels
    means = means.merge(new_VEnames, on='Parcel', how='inner')
    print(means.shape)

(1000,)
(1000, 6)
(66, 11)


In [151]:
means

,Parcel,GluCEST,NMDA,mGluR5,GABA,Lobe,von_economo,VE_Region,VEregion
0,caudalanteriorcingulate_2R,-0.189890,0.106288,1.463622,0.048621,NaN,limbic,limbic,limbic
1,caudalanteriorcingulate_5R,-0.081938,-0.026330,1.229935,-0.473436,NaN,limbic,limbic,limbic
2,cuneus_3R,0.632433,-0.522157,0.053661,0.654213,NaN,primary/secondary sensory,primary/secondary sensory,primary/secondary sensory
3,cuneus_5R,1.114251,0.046650,0.220775,1.635019,NaN,primary/secondary sensory,primary/secondary sensory,primary/secondary sensory
4,cuneus_6R,1.059741,-0.028373,0.215935,1.401222,NaN,primary/secondary sensory,primary/secondary sensory,primary/secondary sensory
...,...,...,...,...,...,...,...,...,...
61,superiorfrontal_39R,-0.593505,-0.539328,0.194108,-0.184789,frontal,association,association1,association1
62,superiorfrontal_40R,-0.803591,-1.089619,0.456369,-0.236281,frontal,association,association1,association1
63,superiorfrontal_7R,-0.408732,0.739121,1.834678,1.177836,frontal,association,association1,association1
64,superiorfrontal_8R,-0.676954,-0.331275,-0.129873,-0.108921,frontal,association,association1,association1


### Functions

In [88]:
# NEW CODE
def calculate_permuted_corrs(merged_dfs, nmaps, region):
    """
    Calculates GluCEST-receptor correlation values within a permuted von_economo region for 
    each null df and stores in output df
    Args:
    null_dfs (list): permutation matrices with new indices. the only thing that matters here is the index 
    and the von economo label. 
    nmaps (list): list of receptors.
    region (string): von economo region.
    Returns:
    null_dfs (list): List of DataFrames with rows reordered according to scrambled indices.
    """ 
    # Initialize an empty dataframe to store the results
    columns = ['shift_iteration']
    for col in nmaps:
        columns.append(f'{col}_r')
        columns.append(f'{col}_p')
    
    null_stats_df = pd.DataFrame(columns=columns) # Make one of these for each von_economo region
    
    # Loop through each null dataframe (each shift iteration)
    for i, merged_df in enumerate(merged_dfs):
        result_row = {f'{region}_permutation': i + 1}
        
        # The rows of null_df are used to index the
        merged_df = merged_df[merged_df['VEregion'] == region]
        
        # For each nmaps column, calculate r and p values with respect to CESTavg
        for col in nmaps:
            r_value, p_value = pearsonr(merged_df['GluCEST'], merged_df[col])
            result_row[f'{col}_r'] = r_value
            result_row[f'{col}_p'] = p_value
        
        # Convert result_row to a DataFrame and concatenate it
        result_row_df = pd.DataFrame([result_row])
        null_stats_df = pd.concat([null_stats_df, result_row_df], ignore_index=True)
    
    return null_stats_df

########################

def calculate_region_correlations(df, nmaps, region):
    """
    This function calculates pearson correlations between CEST_avg and nmap columns.
    Args:
    df (pd.DataFrame): df with cest and nmap parcelwise data
    nmaps (list): names of nmaps
    Returns: 
    Correlation values and p values for parametric associations.
    """
    real_corrs = {}
    df = df[df['VEregion_x']==region]
    
    for col in nmaps:
        #r_value, _ = pearsonr(df['CEST_avg'], df[col])
        r_value, _ = pearsonr(df['GluCEST'], df[col])
        real_corrs[col] = r_value
    return real_corrs

#####################

def calculate_empirical_pvalues(real_corrs, null_stats_df, nmaps, region):
    """
    This function compares real correlations to the null distribution to calculate empirical p-values
    Args:
    real_corrs (pd.DataFrame): df with r and p values for cest-nmap associations
    null_stats_df (pd.DataFrame): df with r and p value for cest-nmap associations per iteration
    nmaps (list): names of nmaps
    Returns: 
    Empirical p values based on null model
    """    
    p_values = {}
    
    # Loop through each column in nmaps
    for col in nmaps:
        # Extract the null r-values for this nmaps column
        null_r_values = null_stats_df[f'{col}_r']
        
        # Calculate the empirical p-value: proportion of null r-values more extreme than the real r-value
        real_r = real_corrs[col]
        
        # Golia's addition - correct for if r values aren't centered on zero:
        null_mean = null_r_values.mean() # Find mean of r nulls
        # Subtract mean from null and real r values.
        null_r_values = null_r_values - null_mean 
        real_r = real_r - null_mean
        
        # Two-tailed test: count both large positive and negative extremes
        extreme_count = np.sum(np.abs(null_r_values) >= np.abs(real_r))
        
        # Compute the empirical p-value
        p_empirical = extreme_count / len(null_r_values)
        p_values[col] = round(p_empirical, 15)
    
    return p_values

#####################

def fisher_z_test(r1, r2, n1, n2):
    # Fisher Z transformation
    z1 = 0.5 * np.log((1 + r1) / (1 - r1))
    z2 = 0.5 * np.log((1 + r2) / (1 - r2))
    
    # Standard error
    se = np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    
    # Z statistic
    z_stat = (z1 - z2) / se
    
    # Two-tailed p-value
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))
    
    return z_stat, p_value

### Generate permutations

In [153]:
# Convert fsavg annotations to gifti
fsgifti = images.annot_to_gifti((annot_lh, annot_rh))
# Spin indices
spin_index = nulls.vazquez_rodriguez(data=None, atlas='fsaverage', density='164k', # Fix double check density '164k'
                                     n_perm=n_permute, seed=1234, parcellation=fsgifti)


In [157]:
# Compile null dfs
test = np.array(receptor_df.iloc[:, 1:])
null = test[spin_index]
null_dfs = list()
for row_index in range(null.shape[1]):  
    spindf = pd.DataFrame(null[:, row_index, :], columns=['NMDA', 'mGluR5', 'GABA','CB1','VEregion'])
    spindf['Parcel'] = labels
    null_dfs.append(spindf)
print(len(null_dfs))

100


In [ ]:
#Fix, why is VEregion column getting duplicated??

In [158]:
# NEW CODE:
# We are creating merged_dfs list to exclude spins containing insular region.

# Create a list to store the merged DataFrames
merged_dfs = []

# Loop through the list of null_dfs
for null_df in null_dfs:
    # Keep only column with region label
    # Keep only parcels in GluCEST slice
    null_df = null_df[['Parcel', 'VEregion']]
    merged_df_new = means.merge(null_df, on='Parcel', how='inner')

    # Exclude merged_dfs that include "insular" region
    if not merged_df_new['VEregion'].str.contains("insular", case=False).any():
        # Append the filtered DataFrame to the list
        merged_dfs.append(merged_df_new)

print(merged_dfs[1])
print(merged_dfs[4])

                        Parcel   GluCEST      NMDA    mGluR5      GABA  \
0   caudalanteriorcingulate_2R -0.189890  0.106288  1.463622  0.048621   
1   caudalanteriorcingulate_5R -0.081938 -0.026330  1.229935 -0.473436   
2                    cuneus_3R  0.632433 -0.522157  0.053661  0.654213   
3                    cuneus_5R  1.114251  0.046650  0.220775  1.635019   
4                    cuneus_6R  1.059741 -0.028373  0.215935  1.401222   
..                         ...       ...       ...       ...       ...   
61         superiorfrontal_39R -0.593505 -0.539328  0.194108 -0.184789   
62         superiorfrontal_40R -0.803591 -1.089619  0.456369 -0.236281   
63          superiorfrontal_7R -0.408732  0.739121  1.834678  1.177836   
64          superiorfrontal_8R -0.676954 -0.331275 -0.129873 -0.108921   
65          superiorfrontal_9R -0.209552 -0.293822 -0.660154 -0.651477   

       Lobe                von_economo                VE_Region_x  \
0       NaN                     limbic    

In [176]:
regions = ['association2', 'association1', 'primary motor', 
           'limbic', 'primary/secondary sensory']

# Calculate null stats per VE region.
for region in regions: 
    null_stats_df = calculate_permuted_corrs(merged_dfs, nmaps, region)
    print(null_stats_df)
    real_corrs = calculate_region_correlations(means, nmaps, region) #Fix add new von economo labels to means.
    print(real_corrs)
    pvals = calculate_empirical_pvalues(real_corrs, null_stats_df, nmaps, region)
    print(pvals)


/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_4647/596596570.py:37: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  null_stats_df = pd.concat([null_stats_df, result_row_df], ignore_index=True)


ValueError: x and y must have length at least 2.

In [20]:
results = []
von_economos = ["association","limbic","primary/secondary sensory","primary motor"]
#von_economos = ["association","limbic","primary motor"] #"primary/secondary sensory",
grouped = False

for von_ec in von_economos:
    
    # Keep only parcels in von economo region within slice
    #means_cest = means[['hstatus', 'CEST_avg']]
    if grouped: 
        means_trimmed = means[means['von_economo']==von_ec]
        means_cest = means[means['von_economo']==von_ec]
        #means_cest = means_cest[['hstatus', 'GluCEST']]
    else:
        means_cest = means[means['von_economo']==von_ec]
        means_cest = means_cest[['GluCEST']]
        means_trimmed = means[means['von_economo']==von_ec]

    # Filter dfs to keep only parcels in slice
    merged_dfs = []
    for null_df in null_dfs:
        merged_df = means_cest.merge(null_df, on='Parcel', how='inner')
        merged_dfs.append(merged_df)
        print(merged_df.shape)

    # Calculate stats of null distributions
    null_stats_df = calculate_null_corrs(merged_dfs, nmaps)
    real_corrs = calculate_real_correlations(means_trimmed, nmaps)
    empirical_p_values = calculate_empirical_pvalues(real_corrs, null_stats_df, nmaps)
    
    # Save results for each von Economo region
    for col, p_val in empirical_p_values.items():
        r_val = real_corrs[col]
        results.append({
            'Receptor': col,
            'von_economo': von_ec,
            'r_value': r_val,
            'p_value': p_val,
            'n_parcels': len(means_trimmed)
        })
spins_df = pd.DataFrame(results)
print(spins_df)


(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(34, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)
(15, 7)


/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_4647/1207369118.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  null_stats_df = pd.concat([null_stats_df, result_row_df], ignore_index=True)
/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_4647/1207369118.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  null_stats_df = pd.concat([null_stats_df, result_row_df], ignore_index=True)


(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(9, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)
(7, 7)

/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_4647/1207369118.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  null_stats_df = pd.concat([null_stats_df, result_row_df], ignore_index=True)
/var/folders/kk/w6xmtt2d55xfbxvb6wcqhq580000gp/T/ipykernel_4647/1207369118.py:31: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  null_stats_df = pd.concat([null_stats_df, result_row_df], ignore_index=True)


In [133]:
spins_df = results_df

In [134]:
# Perform FDR correction
from statsmodels.stats.multitest import multipletests
spins_df['p_value_fdr'], spins_df['fdr_corrected'], _, _ = multipletests(spins_df['p_value'], method='fdr_bh')

# Sort results by FDR-corrected p-values (from most to least significant)
spins_df_sorted = spins_df.sort_values(by='fdr_corrected', ascending=True)

# Display the sorted DataFrame
spins_df_sorted

,Receptor,von_economo,r_value,p_value,n_parcels,p_value_fdr,fdr_corrected
2,GABA,association,0.686427,0.0019,34,True,0.022800
10,mGluR5,primary motor,0.884931,0.0073,7,True,0.043800
11,GABA,primary motor,0.808625,0.0224,7,False,0.089600
8,GABA,primary/secondary sensory,0.819528,0.0340,9,False,0.102000
0,NMDA,association,0.504547,0.0594,34,False,0.142560
1,mGluR5,association,0.531167,0.0724,34,False,0.144800
7,mGluR5,primary/secondary sensory,-0.714701,0.1129,9,False,0.173100
9,NMDA,primary motor,0.658564,0.1154,7,False,0.173100
3,NMDA,limbic,0.255097,0.5052,15,False,0.673600
4,mGluR5,limbic,0.164236,0.6893,15,False,0.751964


In [123]:
import pandas as pd
from itertools import combinations

df = pd.DataFrame(data)

# Function to perform Fisher's t-test
def fisher_t_test(r1, r2, n1, n2):
    """
    Perform Fisher's t-test for two correlations.
    """
    import numpy as np
    from scipy.stats import norm

    def fisher_r_to_z(r):
        return 0.5 * np.log((1 + r) / (1 - r))

    z1 = fisher_r_to_z(r1)
    z2 = fisher_r_to_z(r2)
    se_diff = np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    z_diff = (z1 - z2) / se_diff
    p_value = 2 * (1 - norm.cdf(abs(z_diff)))  # Two-tailed test
    return z_diff, p_value

# Prepare results DataFrame
results = []

# Group by 'Receptor' and run pairwise comparisons
for receptor, group in results_df.groupby('Receptor'):
    rows = group.to_dict(orient='records')
    
    # Pairwise combinations of rows
    for (row1, row2) in combinations(rows, 2):
        r1, r2 = row1['r_value'], row2['r_value']
        n1, n2 = row1['n_parcels'], row2['n_parcels']
        
        # Perform Fisher's t-test
        z_diff, p_val = fisher_t_test(r1, r2, n1, n2)
        
        # Append results
        results.append({
            'Receptor': receptor,
            'von_economo_1': row1['von_economo'],
            'von_economo_2': row2['von_economo'],
            'r_value_1': r1,
            'r_value_2': r2,
            'z_diff': z_diff,
            'p_value': p_val
        })

# Convert results to a DataFrame
results_df = pd.DataFrame(results)

# Save or display results
print(results_df)


   Receptor              von_economo_1              von_economo_2  r_value_1  \
0      GABA                association                     limbic   0.686427   
1      GABA                association  primary/secondary sensory   0.686427   
2      GABA                association              primary motor   0.686427   
3      GABA                     limbic  primary/secondary sensory   0.106708   
4      GABA                     limbic              primary motor   0.106708   
5      GABA  primary/secondary sensory              primary motor   0.819528   
6      NMDA                association                     limbic   0.504547   
7      NMDA                association  primary/secondary sensory   0.504547   
8      NMDA                association              primary motor   0.504547   
9      NMDA                     limbic  primary/secondary sensory   0.255097   
10     NMDA                     limbic              primary motor   0.255097   
11     NMDA  primary/secondary sensory  

In [124]:
# Perform FDR correction
from statsmodels.stats.multitest import multipletests
results_df['p_value_fdr'], results_df['fdr_corrected'], _, _ = multipletests(results_df['p_value'], method='fdr_bh')

# Sort results by FDR-corrected p-values (from most to least significant)
results_df_sorted = results_df.sort_values(by='fdr_corrected', ascending=True)

# Display the sorted DataFrame
results_df_sorted

,Receptor,von_economo_1,von_economo_2,r_value_1,r_value_2,z_diff,p_value,p_value_fdr,fdr_corrected
17,mGluR5,primary/secondary sensory,primary motor,-0.714701,0.884931,-3.555065,0.000378,True,0.006802
13,mGluR5,association,primary/secondary sensory,0.531167,-0.714701,3.337364,0.000846,True,0.007612
15,mGluR5,limbic,primary/secondary sensory,0.164236,-0.714701,2.124929,0.033593,False,0.108106
16,mGluR5,limbic,primary motor,0.164236,0.884931,-2.134441,0.032807,False,0.108106
0,GABA,association,limbic,0.686427,0.106708,2.159055,0.030846,False,0.108106
3,GABA,limbic,primary/secondary sensory,0.106708,0.819528,-2.096528,0.036035,False,0.108106
4,GABA,limbic,primary motor,0.106708,0.808625,-1.759638,0.078469,False,0.201778
14,mGluR5,association,primary motor,0.531167,0.884931,-1.517633,0.129107,False,0.290490
12,mGluR5,association,limbic,0.531167,0.164236,1.253083,0.210176,False,0.420351
6,NMDA,association,limbic,0.504547,0.255097,0.866300,0.386326,False,0.603415


In [21]:
# Calculate stats of null distributions
null_stats_df = calculate_null_corrs(merged_dfs, nmaps)

real_corrs = calculate_real_correlations(means, nmaps)
empirical_p_values = calculate_empirical_pvalues(real_corrs, null_stats_df, nmaps)

print(empirical_p_values)

KeyError: 'CEST_avg'

In [64]:
print("Pearson R values with real data:\n",real_corrs,"\n\nEmpirical p values based on ",n_permute, " permutations:")
for col, p_val in empirical_p_values.items():
    print(f'{col}: {p_val:.25f}') 

Pearson R values with real data:
 {'NMDA': 0.4794093831311077, 'mGluR5': 0.24549237724901335, 'GABA': 0.6757882541171423} 

Empirical p values based on  10000  permutations:
NMDA: 0.0115999999999999992006394
mGluR5: 0.3133000000000000229150032
GABA: 0.0001000000000000000047922


In [50]:
import numpy as np
from scipy.stats import norm


# Example usage
r_hc_nmda = 0.47552583 # Correlation coefficient in HC
r_psy_nmda = 0.4861826   # Correlation coefficient in PSY
r_hc_gaba = 0.6970255086 # Correlation coefficient in HC
r_psy_gaba = 0.65476845  # Correlation coefficient in PSY
n_hc = 36    # Sample size in HC
n_psy = 55   # Sample size in PSY

z_stat, p_value = fisher_z_test(r_hc_nmda, r_psy_nmda, n_hc, n_psy)
print(f"Z-statistic for NMDA: {z_stat}, p-value: {p_value}")
z_stat, p_value = fisher_z_test(r_hc_gaba, r_psy_gaba, n_hc, n_psy)
print(f"Z-statistic for NMDA: {z_stat}, p-value: {p_value}")


Z-statistic for NMDA: -0.062285207013442416, p-value: 0.9503357086961357
Z-statistic for NMDA: 0.3499760378779926, p-value: 0.72635668083412


In [66]:
# Fishers test for lobar comparisons

import numpy as np
from scipy.stats import norm

def fisher_z_test(r1, r2, n1, n2):
    # Fisher Z transformation
    z1 = 0.5 * np.log((1 + r1) / (1 - r1))
    z2 = 0.5 * np.log((1 + r2) / (1 - r2))
    
    # Standard error
    se = np.sqrt(1 / (n1 - 3) + 1 / (n2 - 3))
    
    # Z statistic
    z_stat = (z1 - z2) / se
    
    # Two-tailed p-value
    p_value = 2 * (1 - norm.cdf(abs(z_stat)))
    
    return z_stat, p_value

# Example usage
r_par_nmda = 0.6482948 # Correlation coefficient in HC
r_fro_nmda = 0.4300273  # Correlation coefficient in PSY
r_par_gaba = 0.771351 # Correlation coefficient in HC
r_fro_gaba = 0.08859911  # Correlation coefficient in PSY
n_par = 19   # Sample size in HC
n_fro = 29   # Sample size in PSY

z_stat, p_value = fisher_z_test(r_par_nmda, r_fro_nmda, n_par, n_fro)
print(f"Z-statistic for NMDA: {z_stat}, p-value: {p_value}")
z_stat, p_value = fisher_z_test(r_par_gaba, r_fro_gaba, n_par, n_fro)
print(f"Z-statistic for NMDA: {z_stat}, p-value: {p_value}")


Z-statistic for NMDA: 0.9832475360078208, p-value: 0.32548562426139815
Z-statistic for NMDA: 2.9420586798664785, p-value: 0.0032603813912401858
